In [0]:
print("E-commerce Data Engineering Project")
print("Python environment is working!")

print("Spark version:", spark.version)

E-commerce Data Engineering Project
Python environment is working!
Spark version: 4.2.0


In [0]:
file_path = "/Volumes/workspace/default/ecommerce_raw/customers_messy.csv"

display(dbutils.fs.ls(
    file_path
))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,customers_messy.csv,5556,1791572411000


In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(file_path)
)

display(df)

customer_id,name,city,email,phone,signup_date,status,source_updated_at
C001,ARUN,Chennai,arun.kumar@example.com,+91 98765 43210,2026-01-10,ACTIVE,2026-10-08T09:00:00
C002,Priya,Bangalore,priya.sharma@example.com,9876543210,2026-01-12,active,2026-10-08T09:05:00
C003,Karthik,Hyderabad,karthik@example.com,+91-9988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00
C004,MEENA,Chennai,meena@example.com,(044) 2345 6789,20-01-2026,Active,2026-10-08T09:15:00
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00
C002,Priya,Bengaluru,priya.sharma@example.com,+91 98765 43210,2026-01-12,ACTIVE,2026-10-08T10:00:00
C006,Divya,NULL,divya@example.com,null,2026-02-05,active,2026-10-08T09:25:00
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00
C008,ananya rao,BANGALORE,ananya.rao@example.com,91 99887 76655,2026-02-29,ACTIVE,2026-10-08T09:35:00
C009,null,Delhi,naveen@example.com,98765-43210,2026-03-01,ACTIVE,2026-10-08T09:40:00


In [0]:
# checking source data
from pyspark.sql import functions as F

# 1. Total number of records
print("Total records:", df.count())

# 2. Number of columns
print("Total columns:", len(df.columns))

# 3. Display column names and data types
df.printSchema()

# 4. Count distinct customer IDs
print(
    "Distinct customer IDs:",
    df.select("customer_id").distinct().count()
)

# 5. Show duplicate customer IDs
df.groupBy("customer_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .show()

Total records: 63
Total columns: 8
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_updated_at: string (nullable = true)

Distinct customer IDs: 61
+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C014|    2|
|       C002|    2|
+-----------+-----+



In [0]:
from pyspark.sql import functions as F

df.filter(
    F.col("customer_id").isin("C002", "C014")
).orderBy(
    "customer_id",
    "source_updated_at"
).show(truncate=False)

+-----------+--------+---------+------------------------+---------------+-----------+------+-------------------+
|customer_id|name    |city     |email                   |phone          |signup_date|status|source_updated_at  |
+-----------+--------+---------+------------------------+---------------+-----------+------+-------------------+
|C002       |Priya   |Bangalore|priya.sharma@example.com|9876543210     |2026-01-12 |active|2026-10-08T09:05:00|
|C002       | Priya  |Bengaluru|priya.sharma@example.com|+91 98765 43210|2026-01-12 |ACTIVE|2026-10-08T10:00:00|
|C014       |Sneha   |Mysuru   |sneha@example.com       |9876543214     |2026-03-20 |ACTIVE|2026-10-08T10:10:00|
|C014       |Sneha P.|Mysore   |sneha.p@example.com     |9876543214     |2026-03-20 |ACTIVE|2026-10-09T08:00:00|
+-----------+--------+---------+------------------------+---------------+-----------+------+-------------------+



In [0]:
# Check available catalogs and schemas
display(spark.sql("SHOW CATALOGS"))

catalog
samples
system
teju_first
workspace


In [0]:
# Convert source_updated_at into a timestamp
from pyspark.sql.window import Window
df = df.withColumn(
    "source_updated_at",
    F.to_timestamp("source_updated_at")
)

# Select the latest record for each customer
window_spec = Window.partitionBy("customer_id").orderBy(
    F.col("source_updated_at").desc_nulls_last()
)

silver_customers_df = (
    df.withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)

silver_customers_df.orderBy("customer_id").show(truncate=False)


+-----------+------------+----------+------------------------+---------------+-----------+--------+-------------------+
|customer_id|name        |city      |email                   |phone          |signup_date|status  |source_updated_at  |
+-----------+------------+----------+------------------------+---------------+-----------+--------+-------------------+
|NULL       |Unknown     |Unknown   |unknown@example.com     |0000000000     |not-a-date |ACTIVE  |2026-10-08 13:20:00|
|C001       |  ARUN      |Chennai   |arun.kumar@example.com  |+91 98765 43210|2026-01-10 |ACTIVE  |2026-10-08 09:00:00|
|C002       | Priya      |Bengaluru |priya.sharma@example.com|+91 98765 43210|2026-01-12 |ACTIVE  |2026-10-08 10:00:00|
|C003       |Karthik     |Hyderabad |karthik@example.com     |+91-9988776655 |2026/01/15 |ACTIVE  |2026-10-08 09:10:00|
|C004       |MEENA       |Chennai   |meena@example.com       |(044) 2345 6789|20-01-2026 |Active  |2026-10-08 09:15:00|
|C005       |Rahul       |Mumbai    |rah

In [0]:
# Check that each customer appears only once
silver_customers_df.groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Original records:", df.count())
print("Silver records:", silver_customers_df.count())

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

Original records: 63
Silver records: 61


In [0]:
silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_customers")

In [0]:
display(
    spark.table("workspace.default.silver_customers")
         .orderBy("customer_id")
)

print(
    "Silver table count:",
    spark.table("workspace.default.silver_customers").count()
)

customer_id,name,city,email,phone,signup_date,status,source_updated_at
null,Unknown,Unknown,unknown@example.com,0000000000,not-a-date,ACTIVE,2026-10-08T13:20:00.000Z
C001,ARUN,Chennai,arun.kumar@example.com,+91 98765 43210,2026-01-10,ACTIVE,2026-10-08T09:00:00.000Z
C002,Priya,Bengaluru,priya.sharma@example.com,+91 98765 43210,2026-01-12,ACTIVE,2026-10-08T10:00:00.000Z
C003,Karthik,Hyderabad,karthik@example.com,+91-9988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00.000Z
C004,MEENA,Chennai,meena@example.com,(044) 2345 6789,20-01-2026,Active,2026-10-08T09:15:00.000Z
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00.000Z
C006,Divya,NULL,divya@example.com,null,2026-02-05,active,2026-10-08T09:25:00.000Z
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00.000Z
C008,ananya rao,BANGALORE,ananya.rao@example.com,91 99887 76655,2026-02-29,ACTIVE,2026-10-08T09:35:00.000Z
C009,null,Delhi,naveen@example.com,98765-43210,2026-03-01,ACTIVE,2026-10-08T09:40:00.000Z


Silver table count: 61


In [0]:
from pyspark.sql import functions as F

silver_df = spark.read.table("workspace.default.silver_customers")

# 1. Check duplicate customer IDs
duplicate_count = (
    silver_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 2. Check missing customer IDs
null_id_count = silver_df.filter(
    F.col("customer_id").isNull() |
    (F.trim(F.col("customer_id")) == "")
).count()

# 3. Check missing names
missing_name_count = silver_df.filter(
    F.col("name").isNull() |
    (F.trim(F.col("name")) == "")
).count()

# 4. Check invalid email formats
invalid_email_count = silver_df.filter(
    F.col("email").isNull() |
    (~F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))
).count()

print("Duplicate customer IDs:", duplicate_count)
print("Missing customer IDs:", null_id_count)
print("Missing names:", missing_name_count)
print("Invalid emails:", invalid_email_count)

Duplicate customer IDs: 0
Missing customer IDs: 1
Missing names: 2
Invalid emails: 3


In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("workspace.default.silver_customers")

standardized_df = (
    silver_df

    # Standardize name spacing
    .withColumn(
        "name",
        F.when(
            F.col("name").isNull() |
            (F.trim(F.col("name")) == ""),
            None
        ).otherwise(F.trim(F.col("name")))
    )

    # Standardize email casing and whitespace
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )

    # Standardize status casing
    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )

    # Standardize known city variations
    .withColumn(
        "city",
        F.when(
            F.lower(F.trim(F.col("city"))).isin("bangalore", "bengaluru"),
            "Bengaluru"
        ).when(
            F.lower(F.trim(F.col("city"))).isin("madras", "chennai"),
            "Chennai"
        ).otherwise(F.initcap(F.trim(F.col("city"))))
    )

    # Keep only digits in the phone number
    .withColumn(
        "phone",
        F.regexp_replace(F.col("phone"), r"\D", "")
    )
)

display(standardized_df.orderBy("customer_id"))

customer_id,name,city,email,phone,signup_date,status,source_updated_at
null,Unknown,Unknown,unknown@example.com,0000000000,not-a-date,ACTIVE,2026-10-08T13:20:00.000Z
C001,ARUN,Chennai,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,2026-10-08T09:00:00.000Z
C002,Priya,Bengaluru,priya.sharma@example.com,919876543210,2026-01-12,ACTIVE,2026-10-08T10:00:00.000Z
C003,Karthik,Hyderabad,karthik@example.com,919988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00.000Z
C004,MEENA,Chennai,meena@example.com,04423456789,20-01-2026,ACTIVE,2026-10-08T09:15:00.000Z
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00.000Z
C006,Divya,Null,divya@example.com,null,2026-02-05,ACTIVE,2026-10-08T09:25:00.000Z
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00.000Z
C008,ananya rao,Bengaluru,ananya.rao@example.com,919988776655,2026-02-29,ACTIVE,2026-10-08T09:35:00.000Z
C009,null,Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:40:00.000Z


In [0]:
print("Rows before standardization:", silver_df.count())
print("Rows after standardization:", standardized_df.count())

print(
    "Missing names:",
    standardized_df.filter(F.col("name").isNull()).count()
)

print(
    "Invalid emails:",
    standardized_df.filter(
        F.col("email").isNull() |
        (~F.col("email").rlike(
            r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
        ))
    ).count()
)

Rows before standardization: 61
Rows after standardization: 61
Missing names: 2
Invalid emails: 3


In [0]:
from pyspark.sql import functions as F

# Use the standardized DataFrame from the previous step

email_valid = F.coalesce(
    F.col("email").rlike(
        r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
    ),
    F.lit(False)
)

customer_id_valid = (
    F.col("customer_id").isNotNull() &
    (F.trim(F.col("customer_id")) != "")
)

name_valid = (
    F.col("name").isNotNull() &
    (F.trim(F.col("name")) != "")
)

classified_df = standardized_df.withColumn(
    "is_valid",
    customer_id_valid & name_valid & email_valid
)

valid_customers_df = classified_df.filter(
    F.col("is_valid")
).drop("is_valid")

quarantine_customers_df = classified_df.filter(
    ~F.col("is_valid")
).drop("is_valid")

print("Total records:", classified_df.count())
print("Valid records:", valid_customers_df.count())
print("Quarantined records:", quarantine_customers_df.count())

Total records: 61
Valid records: 56
Quarantined records: 5


In [0]:
display(quarantine_customers_df)


customer_id,name,city,email,phone,signup_date,status,source_updated_at
null,Unknown,Unknown,unknown@example.com,0000000000,not-a-date,ACTIVE,2026-10-08T13:20:00.000Z
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00.000Z
C009,null,Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:40:00.000Z
C021,null,Kolkata,not-an-email,,2026-04-20,ACTIVE,2026-10-08T10:45:00.000Z
C052,Muthu,Chennai,null,9876543252,2026-09-20,ACTIVE,2026-10-08T13:25:00.000Z


In [0]:
quarantine_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.quarantine_customers")

display(spark.table("workspace.default.quarantine_customers"))

customer_id,name,city,email,phone,signup_date,status,source_updated_at
null,Unknown,Unknown,unknown@example.com,0000000000,not-a-date,ACTIVE,2026-10-08T13:20:00.000Z
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00.000Z
C009,null,Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:40:00.000Z
C021,null,Kolkata,not-an-email,,2026-04-20,ACTIVE,2026-10-08T10:45:00.000Z
C052,Muthu,Chennai,null,9876543252,2026-09-20,ACTIVE,2026-10-08T13:25:00.000Z


In [0]:
valid_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_customers")

In [0]:
silver_final_df = spark.table(
    "workspace.default.silver_customers"
)

quarantine_df = spark.table(
    "workspace.default.quarantine_customers"
)

print("Silver valid records:", silver_final_df.count())
print("Quarantined records:", quarantine_df.count())

display(silver_final_df)

Silver valid records: 56
Quarantined records: 5


customer_id,name,city,email,phone,signup_date,status,source_updated_at
C001,ARUN,Chennai,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,2026-10-08T09:00:00.000Z
C002,Priya,Bengaluru,priya.sharma@example.com,919876543210,2026-01-12,ACTIVE,2026-10-08T10:00:00.000Z
C003,Karthik,Hyderabad,karthik@example.com,919988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00.000Z
C004,MEENA,Chennai,meena@example.com,04423456789,20-01-2026,ACTIVE,2026-10-08T09:15:00.000Z
C006,Divya,Null,divya@example.com,null,2026-02-05,ACTIVE,2026-10-08T09:25:00.000Z
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00.000Z
C008,ananya rao,Bengaluru,ananya.rao@example.com,919988776655,2026-02-29,ACTIVE,2026-10-08T09:35:00.000Z
C010,Naveen,New Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:45:00.000Z
C011,Fatima,Chennai,fatima@example.com,9876543211,2026-13-01,ACTIVE,2026-10-08T09:50:00.000Z
C012,Ravi,Kochi,ravi@example.com,9876543212,2026-03-15,UNKNOWN,2026-10-08T09:55:00.000Z


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/ecommerce_raw/customers_messy.csv"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,customers_messy.csv,5556,1791572411000


In [0]:
raw_path = "/Volumes/workspace/default/ecommerce_raw/customers_messy.csv"

raw_customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(raw_path)
)

display(raw_customers_df)

customer_id,name,city,email,phone,signup_date,status,source_updated_at
C001,ARUN,Chennai,arun.kumar@example.com,+91 98765 43210,2026-01-10,ACTIVE,2026-10-08T09:00:00
C002,Priya,Bangalore,priya.sharma@example.com,9876543210,2026-01-12,active,2026-10-08T09:05:00
C003,Karthik,Hyderabad,karthik@example.com,+91-9988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00
C004,MEENA,Chennai,meena@example.com,(044) 2345 6789,20-01-2026,Active,2026-10-08T09:15:00
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00
C002,Priya,Bengaluru,priya.sharma@example.com,+91 98765 43210,2026-01-12,ACTIVE,2026-10-08T10:00:00
C006,Divya,NULL,divya@example.com,null,2026-02-05,active,2026-10-08T09:25:00
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00
C008,ananya rao,BANGALORE,ananya.rao@example.com,91 99887 76655,2026-02-29,ACTIVE,2026-10-08T09:35:00
C009,null,Delhi,naveen@example.com,98765-43210,2026-03-01,ACTIVE,2026-10-08T09:40:00


In [0]:
from pyspark.sql import functions as F

bronze_customers_df = (
    raw_customers_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

In [0]:
bronze_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_customers")

In [0]:
bronze_df = spark.table(
    "workspace.default.bronze_customers"
)

print("Bronze record count:", bronze_df.count())

display(bronze_df)

Bronze record count: 63


customer_id,name,city,email,phone,signup_date,status,source_updated_at,_source_file,_ingested_at,_ingestion_date
C001,ARUN,Chennai,arun.kumar@example.com,+91 98765 43210,2026-01-10,ACTIVE,2026-10-08T09:00:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C002,Priya,Bangalore,priya.sharma@example.com,9876543210,2026-01-12,active,2026-10-08T09:05:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C003,Karthik,Hyderabad,karthik@example.com,+91-9988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C004,MEENA,Chennai,meena@example.com,(044) 2345 6789,20-01-2026,Active,2026-10-08T09:15:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C002,Priya,Bengaluru,priya.sharma@example.com,+91 98765 43210,2026-01-12,ACTIVE,2026-10-08T10:00:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C006,Divya,NULL,divya@example.com,null,2026-02-05,active,2026-10-08T09:25:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C008,ananya rao,BANGALORE,ananya.rao@example.com,91 99887 76655,2026-02-29,ACTIVE,2026-10-08T09:35:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C009,null,Delhi,naveen@example.com,98765-43210,2026-03-01,ACTIVE,2026-10-08T09:40:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14


In [0]:
bronze_df = spark.table("workspace.default.bronze_customers")

print("Bronze records:", spark.table(
    "workspace.default.bronze_customers"
).count())
print("Silver records:", spark.table(
    "workspace.default.silver_customers"
).count())
print("Quarantine records:", spark.table(
    "workspace.default.quarantine_customers"
).count())

display(
    bronze_df.select(
        "customer_id",
        "name",
        "email",
        "_source_file",
        "_ingested_at"
    )
)

Bronze records: 63
Silver records: 56
Quarantine records: 5


customer_id,name,email,_source_file,_ingested_at
C001,ARUN,arun.kumar@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C002,Priya,priya.sharma@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C003,Karthik,karthik@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C004,MEENA,meena@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C005,Rahul,rahul.example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C002,Priya,priya.sharma@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C006,Divya,divya@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C007,Sanjay,sanjay@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C008,ananya rao,ananya.rao@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z
C009,null,naveen@example.com,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def transform_customers(bronze_df):

    # Standardize customer fields
    df = (
        bronze_df
        .withColumn("name", F.trim(F.col("name")))
        .withColumn("email", F.lower(F.trim(F.col("email"))))
        .withColumn("status", F.upper(F.trim(F.col("status"))))
        .withColumn(
            "city",
            F.when(
                F.lower(F.trim(F.col("city"))).isin(
                    "bangalore", "bengaluru"
                ),
                "Bengaluru"
            ).when(
                F.lower(F.trim(F.col("city"))).isin(
                    "madras", "chennai"
                ),
                "Chennai"
            ).otherwise(F.initcap(F.trim(F.col("city"))))
        )
        .withColumn(
            "phone",
            F.regexp_replace(F.col("phone"), r"\D", "")
        )
    )

    # Deduplicate using the latest source update
    window_spec = Window.partitionBy("customer_id").orderBy(
        F.to_timestamp("source_updated_at").desc_nulls_last()
    )

    return (
        df
        .withColumn("_row_num", F.row_number().over(window_spec))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )

transformed_customers_df = transform_customers(bronze_df)

print("Transformed customer records:", transformed_customers_df.count())
display(transformed_customers_df)

Transformed customer records: 61


customer_id,name,city,email,phone,signup_date,status,source_updated_at,_source_file,_ingested_at,_ingestion_date
null,Unknown,Unknown,unknown@example.com,0000000000,not-a-date,ACTIVE,2026-10-08T13:20:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C001,ARUN,Chennai,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,2026-10-08T09:00:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C002,Priya,Bengaluru,priya.sharma@example.com,919876543210,2026-01-12,ACTIVE,2026-10-08T10:00:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C003,Karthik,Hyderabad,karthik@example.com,919988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C004,MEENA,Chennai,meena@example.com,04423456789,20-01-2026,ACTIVE,2026-10-08T09:15:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C005,Rahul,Mumbai,rahul.example.com,9000011111,2026-02-01,ACTIVE,2026-10-08T09:20:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C006,Divya,Null,divya@example.com,null,2026-02-05,ACTIVE,2026-10-08T09:25:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C008,ananya rao,Bengaluru,ananya.rao@example.com,919988776655,2026-02-29,ACTIVE,2026-10-08T09:35:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14
C009,null,Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:40:00,dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14


In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("workspace.default.silver_customers")

gold_customer_summary = (
    silver_df.groupBy("city", "status")
    .agg(
        F.countDistinct("customer_id").alias("total_customers")
    )
    .orderBy("city", "status")
    
)

display(gold_customer_summary)

city,status,total_customers
Ahmedabad,ACTIVE,1
Bengaluru,ACTIVE,11
Chennai,ACTIVE,17
Coimbatore,ACTIVE,2
Delhi,ACTIVE,2
Goa,ACTIVE,1
Hyderabad,ACTIVE,4
Hyderabad,DELETED,1
Jaipur,ACTIVE,1
Kochi,ACTIVE,1


In [0]:
gold_customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_customer_summary")

In [0]:
gold_df = spark.table(
    "workspace.default.gold_customer_summary"
)

display(gold_df)

print("Gold summary rows:", gold_df.count())

city,status,total_customers
Ahmedabad,ACTIVE,1
Bengaluru,ACTIVE,11
Chennai,ACTIVE,17
Coimbatore,ACTIVE,2
Delhi,ACTIVE,2
Goa,ACTIVE,1
Hyderabad,ACTIVE,4
Hyderabad,DELETED,1
Jaipur,ACTIVE,1
Kochi,ACTIVE,1


Gold summary rows: 20


In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("workspace.default.silver_customers")

scd_initial_df = (
    silver_df
    .withColumn("effective_start_date", F.current_timestamp())
    .withColumn(
        "effective_end_date",
        F.lit(None).cast("timestamp")
    )
    .withColumn("is_current", F.lit(True))
)

scd_initial_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_customers_history")

In [0]:
history_df = spark.table(
    "workspace.default.dim_customers_history"
)

display(
    history_df
)

print("Initial history records:", history_df.count())

customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current
C001,ARUN,Chennai,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,2026-10-08T09:00:00.000Z,2026-11-14T05:28:45.277Z,null,true
C002,Priya,Bengaluru,priya.sharma@example.com,919876543210,2026-01-12,ACTIVE,2026-10-08T10:00:00.000Z,2026-11-14T05:28:45.277Z,null,true
C003,Karthik,Hyderabad,karthik@example.com,919988776655,2026/01/15,ACTIVE,2026-10-08T09:10:00.000Z,2026-11-14T05:28:45.277Z,null,true
C004,MEENA,Chennai,meena@example.com,04423456789,20-01-2026,ACTIVE,2026-10-08T09:15:00.000Z,2026-11-14T05:28:45.277Z,null,true
C006,Divya,Null,divya@example.com,null,2026-02-05,ACTIVE,2026-10-08T09:25:00.000Z,2026-11-14T05:28:45.277Z,null,true
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00.000Z,2026-11-14T05:28:45.277Z,null,true
C008,ananya rao,Bengaluru,ananya.rao@example.com,919988776655,2026-02-29,ACTIVE,2026-10-08T09:35:00.000Z,2026-11-14T05:28:45.277Z,null,true
C010,Naveen,New Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:45:00.000Z,2026-11-14T05:28:45.277Z,null,true
C011,Fatima,Chennai,fatima@example.com,9876543211,2026-13-01,ACTIVE,2026-10-08T09:50:00.000Z,2026-11-14T05:28:45.277Z,null,true
C012,Ravi,Kochi,ravi@example.com,9876543212,2026-03-15,UNKNOWN,2026-10-08T09:55:00.000Z,2026-11-14T05:28:45.277Z,null,true


Initial history records: 56


In [0]:
from pyspark.sql import functions as F

history_df = spark.table(
    "workspace.default.dim_customers_history"
)

   

In [0]:
display(
    spark.sql("SHOW TABLES IN workspace.default")
)

database,tableName,isTemporary
default,bronze_customer_cdc,false
default,bronze_customers,false
default,dim_customers_history,false
default,dim_customers_history_test,false
default,gold_customer_summary,false
default,processed_cdc_events,false
default,quarantine_customers,false
default,silver_customer_cdc,false
default,silver_customers,false


In [0]:
bronze_df = spark.table(
    "workspace.default.bronze_customers"
)

print("Total Bronze records:", bronze_df.count())

display(
    bronze_df.select(
        "_source_file",
        "_ingested_at",
        "_ingestion_date"
    ).distinct()
)

Total Bronze records: 63


_source_file,_ingested_at,_ingestion_date
dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,2026-11-14T05:28:31.606Z,2026-11-14


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/ecommerce_raw/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ecommerce_raw/_checkpoints/,_checkpoints/,0,1794634131995
dbfs:/Volumes/workspace/default/ecommerce_raw/_schemas/,_schemas/,0,1794634131995
dbfs:/Volumes/workspace/default/ecommerce_raw/cdc_incoming/,cdc_incoming/,0,1794634131995
dbfs:/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,customer_cdc_batch_001.json,964,1792913457000
dbfs:/Volumes/workspace/default/ecommerce_raw/customers_messy.csv,customers_messy.csv,5556,1791572411000


In [0]:
from pyspark.sql import functions as F
import json

# Read existing current customer records
current_customers = (
    spark.table("workspace.default.dim_customers_history")
    .filter(F.col("is_current") == True)
    .select(
        "customer_id", "name", "city",
        "email", "phone", "signup_date", "status"
    )
    .orderBy("customer_id")
    .limit(3)
    .collect()
)

if len(current_customers) < 3:
    raise ValueError("Need at least 3 current customers for this test.")

customers = [row.asDict() for row in current_customers]

# Convert date values to JSON-friendly strings
for customer in customers:
    if customer["signup_date"] is not None:
        customer["signup_date"] = str(customer["signup_date"])

# Use real existing IDs for update/delete tests and a new ID for insert
cdc_events = [
    {
        **customers[0],
        "operation": "UPDATE",
        "city": "Coimbatore",
        "source_updated_at": "2026-10-25T10:00:00"
    },
    {
        **customers[1],
        "operation": "UPDATE",
        "status": "INACTIVE",
        "source_updated_at": "2026-10-25T10:01:00"
    },
    {
        "customer_id": "C999",
        "name": "Test Customer",
        "city": "Chennai",
        "email": "test.customer999@example.com",
        "phone": "9000000099",
        "signup_date": "2026-10-25",
        "status": "ACTIVE",
        "operation": "INSERT",
        "source_updated_at": "2026-10-25T10:02:00"
    },
    {
        **customers[2],
        "operation": "DELETE",
        "source_updated_at": "2026-10-25T10:03:00"
    }
]

# Write one JSON object per line (JSONL)
output_path = (
    "/Volumes/workspace/default/ecommerce_raw/"
    "customer_cdc_batch_001.json"
)

json_lines = "\n".join(
    json.dumps(event, default=str)
    for event in cdc_events
)

dbutils.fs.put(output_path, json_lines, overwrite=False)

print("CDC batch created:", output_path)
print("Number of events:", len(cdc_events))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4929243304476635>, line 70
     60 output_path = (
     61     "/Volumes/workspace/default/ecommerce_raw/"
     62     "customer_cdc_batch_001.json"
     63 )
     65 json_lines = "\n".join(
     66     json.dumps(event, default=str)
     67     for event in cdc_events
     68 )
---> 70 dbutils.fs.put(output_path, json_lines, overwrite=False)
     72 print("CDC batch created:", output_path)
     73 print("Number of events:", len(cdc_events))

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:57, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     53     pass
     55 error_exception = ExecutionError(str(e))
---> 57 raise patch_exception_with_error_details(
     58     error_exception,
     59     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore

In [0]:
display(
    spark.read
    .option("multiLine", "false")
    .json(
        "/Volumes/workspace/default/ecommerce_raw/"
        "customer_cdc_batch_001.json"
    )
)

city,customer_id,email,name,operation,phone,signup_date,source_updated_at,status
Coimbatore,C001,arun.kumar@example.com,ARUN,UPDATE,919876543210,2026-01-10,2026-10-25T10:00:00,ACTIVE
Bengaluru,C002,priya.sharma@example.com,Priya,UPDATE,919876543210,2026-01-12,2026-10-25T10:01:00,INACTIVE
Chennai,C999,test.customer999@example.com,Test Customer,INSERT,9000000099,2026-10-25,2026-10-25T10:02:00,ACTIVE
Hyderabad,C003,karthik@example.com,Karthik,DELETE,919988776655,2026/01/15,2026-10-25T10:03:00,ACTIVE


In [0]:
dbutils.fs.mkdirs(
    "/Volumes/workspace/default/ecommerce_raw/cdc_incoming/"
)
dbutils.fs.cp(
    "/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json",
    "/Volumes/workspace/default/ecommerce_raw/cdc_incoming/")

True

In [0]:
from pyspark.sql import functions as F

source_path = (
    "/Volumes/workspace/default/ecommerce_raw/cdc_incoming/"
)

schema_path = (
    "/Volumes/workspace/default/ecommerce_raw/_schemas/customer_cdc/"
)

checkpoint_path = (
    "/Volumes/workspace/default/ecommerce_raw/_checkpoints/customer_cdc/"
)

cdc_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(source_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

query = (
    cdc_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.default.bronze_customer_cdc")
)



print("CDC ingestion completed.")

CDC ingestion completed.


In [0]:
display(
    spark.table("workspace.default.bronze_customer_cdc")
    .select(
        "customer_id",
        "operation",
        "source_updated_at",
        "_source_file",
        "_ingested_at"
    )
    .orderBy("source_updated_at")
)

customer_id,operation,source_updated_at,_source_file,_ingested_at
C001,UPDATE,2026-10-25T10:00:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z
C001,UPDATE,2026-10-25T10:00:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C002,UPDATE,2026-10-25T10:01:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C002,UPDATE,2026-10-25T10:01:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z
C999,INSERT,2026-10-25T10:02:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z
C999,INSERT,2026-10-25T10:02:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C003,DELETE,2026-10-25T10:03:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C003,DELETE,2026-10-25T10:03:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z


In [0]:
from pyspark.sql import functions as F

cdc_bronze = spark.table(
    "workspace.default.bronze_customer_cdc"
)

print("Total CDC events:", cdc_bronze.count())

display(
    cdc_bronze.groupBy("operation").count()
)

display(
    cdc_bronze.select(
        "customer_id",
        "operation",
        "source_updated_at",
        "_source_file",
        "_ingested_at"
    ).orderBy("source_updated_at")
)

Total CDC events: 8


operation,count
DELETE,2
UPDATE,4
INSERT,2


customer_id,operation,source_updated_at,_source_file,_ingested_at
C001,UPDATE,2026-10-25T10:00:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z
C001,UPDATE,2026-10-25T10:00:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C002,UPDATE,2026-10-25T10:01:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C002,UPDATE,2026-10-25T10:01:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z
C999,INSERT,2026-10-25T10:02:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z
C999,INSERT,2026-10-25T10:02:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C003,DELETE,2026-10-25T10:03:00,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,2026-10-24T15:53:38.472Z
C003,DELETE,2026-10-25T10:03:00,/Volumes/workspace/default/ecommerce_raw/customer_cdc_batch_001.json,2026-10-24T11:59:48.555Z


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Bronze CDC events
cdc_bronze = spark.table(
    "workspace.default.bronze_customer_cdc"
)

# Normalize operation and timestamp
cdc_prepared = (
    cdc_bronze
    .withColumn(
        "operation",
        F.upper(F.trim(F.col("operation")))
    )
    .withColumn(
        "source_updated_at",
        F.to_timestamp("source_updated_at")
    )
)

# Identify invalid events
invalid_events = cdc_prepared.filter(
    F.col("customer_id").isNull()
    | F.col("operation").isNull()
    | ~F.col("operation").isin("INSERT", "UPDATE", "DELETE")
    | F.col("source_updated_at").isNull()
)

print("Total Bronze events:", cdc_prepared.count())
print("Invalid events:", invalid_events.count())

display(invalid_events)

Total Bronze events: 8
Invalid events: 0


city,customer_id,email,name,operation,phone,signup_date,source_updated_at,status,_rescued_data,_ingested_at,_ingestion_date,_source_file


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read and prepare Bronze CDC events
cdc_bronze = (
    spark.table("workspace.default.bronze_customer_cdc")
    .withColumn(
        "operation",
        F.upper(F.trim(F.col("operation")))
    )
    .withColumn(
        "source_updated_at",
        F.to_timestamp("source_updated_at")
    )
)

# Keep the latest event per customer for this batch
event_window = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("source_updated_at").desc())
)

silver_cdc = (
    cdc_bronze
    .withColumn("_row_num", F.row_number().over(event_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

# Save to a separate Silver CDC table
(
    silver_cdc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_customer_cdc")
)

print("Silver CDC records:", silver_cdc.count())

display(
    spark.table("workspace.default.silver_customer_cdc")
    .select("customer_id", "operation", "source_updated_at")
    .orderBy("source_updated_at")
)

Silver CDC records: 4


customer_id,operation,source_updated_at
C001,UPDATE,2026-10-25T10:00:00.000Z
C002,UPDATE,2026-10-25T10:01:00.000Z
C999,INSERT,2026-10-25T10:02:00.000Z
C003,DELETE,2026-10-25T10:03:00.000Z


In [0]:
from pyspark.sql import functions as F

silver_cdc = spark.table(
    "workspace.default.silver_customer_cdc"
)

print("Silver CDC total:", silver_cdc.count())

display(
    silver_cdc
    .groupBy("operation")
    .count()
)

display(
    silver_cdc.select(
        "customer_id",
        "operation",
        "source_updated_at"
    ).orderBy("source_updated_at")
)

Silver CDC total: 4


operation,count
UPDATE,2
DELETE,1
INSERT,1


customer_id,operation,source_updated_at
C001,UPDATE,2026-10-25T10:00:00.000Z
C002,UPDATE,2026-10-25T10:01:00.000Z
C999,INSERT,2026-10-25T10:02:00.000Z
C003,DELETE,2026-10-25T10:03:00.000Z


In [0]:
from pyspark.sql import functions as F

silver_cdc = spark.table(
    "workspace.default.silver_customer_cdc"
)

display(
    silver_cdc
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

operation,count
DELETE,1
INSERT,1
UPDATE,2


In [0]:
from pyspark.sql import functions as F

silver_cdc = spark.table(
    "workspace.default.silver_customer_cdc"
)

# Check duplicate customer events in this batch
duplicate_events = (
    silver_cdc
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

# Check valid operation types
invalid_operations = silver_cdc.filter(
    F.col("operation").isNull()
    | ~F.col("operation").isin("INSERT", "UPDATE", "DELETE")
)

print("Silver CDC events:", silver_cdc.count())
print("Duplicate customer events:", duplicate_events.count())
print("Invalid operations:", invalid_operations.count())

display(duplicate_events)
display(invalid_operations)

Silver CDC events: 4
Duplicate customer events: 0
Invalid operations: 0


customer_id,count


city,customer_id,email,name,operation,phone,signup_date,source_updated_at,status,_rescued_data,_ingested_at,_ingestion_date,_source_file


In [0]:
from pyspark.sql import functions as F

TARGET = "workspace.default.dim_customers_history"

history_df = spark.table(TARGET)

current = history_df.filter(F.col("is_current") == True)
source = spark.table("workspace.default.silver_customer_cdc")

tracked_columns = [
    "name", "city", "email", "phone", "signup_date", "status"
]

business_columns = [
    "customer_id", "name", "city", "email",
    "phone", "signup_date", "status"
]

# Validate that every event has a current customer state
current_ids = current.select("customer_id").distinct()

insert_conflicts = (
    source.filter(F.col("operation") == "INSERT")
    .join(current_ids, "customer_id", "inner")
)

orphan_updates = (
    source.filter(F.col("operation") == "UPDATE")
    .join(current_ids, "customer_id", "left_anti")
)

orphan_deletes = (
    source.filter(F.col("operation") == "DELETE")
    .join(current_ids, "customer_id", "left_anti")
)

if any(df.limit(1).count() > 0 for df in [
    insert_conflicts, orphan_updates, orphan_deletes
]):
    raise ValueError(
        "CDC conflicts found. Inspect insert_conflicts, "
        "orphan_updates, and orphan_deletes before merging."
    )

# Compare source updates with current target records
s = source.alias("s")
t = current.alias("t")

changed_condition = F.lit(False)

for c in tracked_columns:
    changed_condition = changed_condition | (
        ~F.col(f"s.{c}").eqNullSafe(F.col(f"t.{c}"))
    )

changed = (
    s.join(t, F.col("s.customer_id") == F.col("t.customer_id"))
    .filter(
        (F.col("s.operation") == "UPDATE") & changed_condition
    )
    .select("s.*")
)

# New customers and delete events
new_customers = (
    source.filter(F.col("operation") == "INSERT")
    .join(current_ids, "customer_id", "left_anti")
)

delete_rows = (
    s.join(t, F.col("s.customer_id") == F.col("t.customer_id"))
    .filter(F.col("s.operation") == "DELETE")
    .select("s.*")
)

# Stage expiration rows
expire_updates = (
    changed
    .withColumn("_merge_key", F.col("customer_id"))
    .withColumn("_action", F.lit("EXPIRE"))
)

expire_deletes = (
    delete_rows
    .withColumn("_merge_key", F.col("customer_id"))
    .withColumn("_action", F.lit("DELETE"))
)

# Stage insertion rows
insert_updates = (
    changed
    .withColumn("_merge_key", F.lit(None).cast("string"))
    .withColumn("_action", F.lit("INSERT"))
)

insert_new = (
    new_customers
    .withColumn("_merge_key", F.lit(None).cast("string"))
    .withColumn("_action", F.lit("INSERT"))
)

staged = (
    expire_updates
    .unionByName(expire_deletes)
    .unionByName(insert_updates)
    .unionByName(insert_new)
)

print("Changed updates:", changed.count())
print("New customers:", new_customers.count())
print("Deletes:", delete_rows.count())
print("Staged actions:", staged.count())

display(
    staged.select(
        "_merge_key", "customer_id", "operation",
        "_action", "source_updated_at"
    ).orderBy("source_updated_at", "_action")
)

Changed updates: 2
New customers: 1
Deletes: 1
Staged actions: 6


_merge_key,customer_id,operation,_action,source_updated_at
C001,C001,UPDATE,EXPIRE,2026-10-25T10:00:00.000Z
null,C001,UPDATE,INSERT,2026-10-25T10:00:00.000Z
C002,C002,UPDATE,EXPIRE,2026-10-25T10:01:00.000Z
null,C002,UPDATE,INSERT,2026-10-25T10:01:00.000Z
null,C999,INSERT,INSERT,2026-10-25T10:02:00.000Z
C003,C003,DELETE,DELETE,2026-10-25T10:03:00.000Z


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.dim_customers_history"
)

insert_values = {
    c: f"s.`{c}`"
    for c in business_columns
}

insert_values.update({
    "effective_start_date": "s.source_updated_at",
    "effective_end_date": "CAST(NULL AS TIMESTAMP)",
    "is_current": "true"
})

(
    target.alias("t")
    .merge(
        staged.alias("s"),
        """
        t.customer_id = s._merge_key
        AND t.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="s._action IN ('EXPIRE', 'DELETE')",
        set={
            "is_current": "false",
            "effective_end_date": "s.source_updated_at"
        }
    )
    .whenNotMatchedInsert(
        condition="s._action = 'INSERT'",
        values=insert_values
    )
    .execute()
)

print("CDC batch merged successfully.")

CDC batch merged successfully.


In [0]:
history_after = spark.table(
    "workspace.default.dim_customers_history"
)

print("Total history rows:", history_after.count())

display(
    history_after
    .filter(F.col("customer_id").isin("C999"))
    .orderBy("effective_start_date")
)

display(
    history_after
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

Total history rows: 59


customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current
C999,Test Customer,Chennai,test.customer999@example.com,9000000099,2026-10-25,ACTIVE,null,2026-10-25T10:02:00.000Z,null,true


customer_id,count


In [0]:
from pyspark.sql import functions as F

history = spark.table("workspace.default.dim_customers_history")

print("Total history rows:", history.count())

print("Current records by customer:")
display(
    history
    .filter(F.col("customer_id").isin("C999"))
    .orderBy("effective_start_date")
)

print("Customers with multiple current records:")
display(
    history
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Current record count:", history.filter(F.col("is_current") == True).count())
print("Expired record count:", history.filter(F.col("is_current") == False).count())

Total history rows: 59
Current records by customer:


customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current
C999,Test Customer,Chennai,test.customer999@example.com,9000000099,2026-10-25,ACTIVE,null,2026-10-25T10:02:00.000Z,null,true


Customers with multiple current records:


customer_id,count


Current record count: 56
Expired record count: 3


In [0]:
from pyspark.sql import functions as F

cdc = spark.table("workspace.default.silver_customer_cdc")

print("CDC schema:")
cdc.printSchema()

print("CDC operation counts:")
display(
    cdc.groupBy("operation")
       .count()
       .orderBy("operation")
)

print("CDC sample records:")
display(cdc.limit(10))

CDC schema:
root
 |-- city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- name: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- _source_file: string (nullable = true)

CDC operation counts:


operation,count
DELETE,1
INSERT,1
UPDATE,2


CDC sample records:


city,customer_id,email,name,operation,phone,signup_date,source_updated_at,status,_rescued_data,_ingested_at,_ingestion_date,_source_file
Coimbatore,C001,arun.kumar@example.com,ARUN,UPDATE,919876543210,2026-01-10,2026-10-25T10:00:00.000Z,ACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json
Bengaluru,C002,priya.sharma@example.com,Priya,UPDATE,919876543210,2026-01-12,2026-10-25T10:01:00.000Z,INACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json
Hyderabad,C003,karthik@example.com,Karthik,DELETE,919988776655,2026/01/15,2026-10-25T10:03:00.000Z,ACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json
Chennai,C999,test.customer999@example.com,Test Customer,INSERT,9000000099,2026-10-25,2026-10-25T10:02:00.000Z,ACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json


In [0]:
from pyspark.sql import functions as F

cdc = spark.table("workspace.default.silver_customer_cdc")

print("Operation counts:")
display(
    cdc.groupBy("operation")
       .count()
       .orderBy("operation")
)

print("Duplicate customer/timestamp combinations:")
display(
    cdc.groupBy("customer_id", "source_updated_at")
       .count()
       .filter(F.col("count") > 1)
)

print("Target table schema:")
spark.table("workspace.default.dim_customers_history").printSchema()

Operation counts:


operation,count
DELETE,1
INSERT,1
UPDATE,2


Duplicate customer/timestamp combinations:


customer_id,source_updated_at,count


Target table schema:
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- effective_start_date: timestamp (nullable = true)
 |-- effective_end_date: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)



In [0]:
from pyspark.sql import functions as F

cdc = spark.table("workspace.default.silver_customer_cdc")

cdc_with_event_key = cdc.withColumn(
    "event_key",
    F.sha2(
        F.to_json(
            F.struct(
                F.col("customer_id"),
                F.col("operation"),
                F.col("source_updated_at"),
                F.col("name"),
                F.col("city"),
                F.col("email"),
                F.col("phone"),
                F.col("signup_date"),
                F.col("status")
            )
        ),
        256
    )
)

print("Total CDC events:", cdc_with_event_key.count())
print(
    "Distinct event keys:",
    cdc_with_event_key.select("event_key").distinct().count()
)

display(
    cdc_with_event_key.select(
        "event_key",
        "customer_id",
        "operation",
        "source_updated_at"
    )
)

Total CDC events: 4
Distinct event keys: 4


event_key,customer_id,operation,source_updated_at
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)

table_name = "workspace.default.processed_cdc_events"

ledger_schema = StructType([
    StructField("event_key", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("source_updated_at", TimestampType(), True),
    StructField("processing_status", StringType(), False),
    StructField("processed_at", TimestampType(), True)
])

if spark.catalog.tableExists(table_name):
    print(f"Ledger table already exists: {table_name}")
else:
    empty_ledger = spark.createDataFrame([], ledger_schema)

    empty_ledger.write \
        .format("delta") \
        .saveAsTable(table_name)

    print(f"Ledger table created: {table_name}")

Ledger table already exists: workspace.default.processed_cdc_events


In [0]:
from pyspark.sql import functions as F

cdc = spark.table("workspace.default.silver_customer_cdc")

events = cdc.withColumn(
    "event_key",
    F.sha2(
        F.to_json(
            F.struct(
                "customer_id",
                "operation",
                "source_updated_at",
                "name",
                "city",
                "email",
                "phone",
                "signup_date",
                "status"
            )
        ),
        256
    )
)

print("Total events:", events.count())
print("Distinct event keys:", events.select("event_key").distinct().count())

display(
    events.select(
        "event_key",
        "customer_id",
        "operation",
        "source_updated_at"
    )
)

Total events: 4
Distinct event keys: 4


event_key,customer_id,operation,source_updated_at
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z


In [0]:
from pyspark.sql import functions as F

events = events.persist()

ledger = spark.table("workspace.default.processed_cdc_events")

new_events = events.join(
    ledger.select("event_key"),
    on="event_key",
    how="left_anti"
)

print("Events in current batch:", events.count())
print("Already recorded in ledger:", events.count() - new_events.count())
print("Not yet recorded:", new_events.count())

display(
    new_events.select(
        "customer_id",
        "operation",
        "source_updated_at",
        "event_key"
    )
)

Events in current batch: 4
Already recorded in ledger: 4
Not yet recorded: 0


customer_id,operation,source_updated_at,event_key


In [0]:
batch_id = (
    events
    .agg(
        F.sha2(
            F.concat_ws(
                "|",
                F.sort_array(F.collect_list("event_key"))
            ),
            256
        ).alias("batch_id")
    )
    .first()["batch_id"]
)

print("CDC batch ID:", batch_id)

CDC batch ID: 583a48c40134345d3c0b9ec5975d56193181cec2ee81bd14916784d1b94842b2


In [0]:
ledger = spark.table("workspace.default.processed_cdc_events")

ledger.printSchema()
print("Existing ledger records:", ledger.count())
display(ledger.limit(5))

root
 |-- event_key: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- processing_status: string (nullable = true)
 |-- processed_at: timestamp (nullable = true)

Existing ledger records: 4


event_key,customer_id,operation,source_updated_at,processing_status,processed_at
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

LEDGER = "workspace.default.processed_cdc_events"

cdc = spark.table("workspace.default.silver_customer_cdc")

events = cdc.withColumn(
    "event_key",
    F.sha2(
        F.to_json(
            F.struct(
                "customer_id",
                "operation",
                "source_updated_at",
                "name",
                "city",
                "email",
                "phone",
                "signup_date",
                "status"
            )
        ),
        256
    )
)

ledger_rows = events.select(
    "event_key",
    "customer_id",
    "operation",
    "source_updated_at"
).withColumn(
    "processing_status", F.lit("PENDING")
).withColumn(
    "processed_at", F.lit(None).cast("timestamp")
)

ledger = DeltaTable.forName(spark, LEDGER)

(
    ledger.alias("t")
    .merge(ledger_rows.alias("s"), "t.event_key = s.event_key")
    .whenNotMatchedInsertAll()
    .execute()
)

print("Ledger registration complete.")
display(
    spark.table(LEDGER)
    .groupBy("processing_status")
    .count()
)

Ledger registration complete.


processing_status,count
PROCESSED,4


In [0]:
from pyspark.sql import functions as F

ledger = spark.table("workspace.default.processed_cdc_events")

print("Total ledger records:", ledger.count())

print("Status counts:")
display(
    ledger.groupBy("processing_status").count()
)

print("Duplicate event keys:")
display(
    ledger.groupBy("event_key")
          .count()
          .filter(F.col("count") > 1)
)

print("Registered events:")
display(
    ledger.select(
        "customer_id",
        "operation",
        "source_updated_at",
        "processing_status"
    ).orderBy("source_updated_at")
)

Total ledger records: 4
Status counts:


processing_status,count
PROCESSED,4


Duplicate event keys:


event_key,count


Registered events:


customer_id,operation,source_updated_at,processing_status
C001,UPDATE,2026-10-25T10:00:00.000Z,PROCESSED
C002,UPDATE,2026-10-25T10:01:00.000Z,PROCESSED
C999,INSERT,2026-10-25T10:02:00.000Z,PROCESSED
C003,DELETE,2026-10-25T10:03:00.000Z,PROCESSED


In [0]:
from pyspark.sql import functions as F

ledger = spark.table("workspace.default.processed_cdc_events")
history = spark.table("workspace.default.dim_customers_history")
display(ledger)
display(history)
pending = ledger.filter(F.col("processing_status") == "PENDING")

current = history.filter(F.col("is_current") == True)

print("Pending events:", pending.count())

print("Pending events by operation:")
display(
    pending.groupBy("operation").count()
)

print("Current history records for affected customers:")
display(
    current.join(
        pending.select("customer_id").distinct(),
        on="customer_id",
        how="inner"
    ).orderBy("customer_id")
)

print("History versions for affected customers:")
display(
    history.join(
        pending.select("customer_id").distinct(),
        on="customer_id",
        how="inner"
    ).orderBy("customer_id", "effective_start_date")
)

event_key,customer_id,operation,source_updated_at,processing_status,processed_at
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z


customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current
C004,MEENA,Chennai,meena@example.com,04423456789,20-01-2026,ACTIVE,2026-10-08T09:15:00.000Z,2026-11-14T05:28:45.277Z,null,true
C006,Divya,Null,divya@example.com,null,2026-02-05,ACTIVE,2026-10-08T09:25:00.000Z,2026-11-14T05:28:45.277Z,null,true
C007,Sanjay,Pune,sanjay@example.com,9123456780,2026-02-10,INACTIVE,2026-10-08T09:30:00.000Z,2026-11-14T05:28:45.277Z,null,true
C008,ananya rao,Bengaluru,ananya.rao@example.com,919988776655,2026-02-29,ACTIVE,2026-10-08T09:35:00.000Z,2026-11-14T05:28:45.277Z,null,true
C010,Naveen,New Delhi,naveen@example.com,9876543210,2026-03-01,ACTIVE,2026-10-08T09:45:00.000Z,2026-11-14T05:28:45.277Z,null,true
C011,Fatima,Chennai,fatima@example.com,9876543211,2026-13-01,ACTIVE,2026-10-08T09:50:00.000Z,2026-11-14T05:28:45.277Z,null,true
C012,Ravi,Kochi,ravi@example.com,9876543212,2026-03-15,UNKNOWN,2026-10-08T09:55:00.000Z,2026-11-14T05:28:45.277Z,null,true
C013,Mohan,Coimbatore,mohan@example.com,9876543213,2027-01-01,ACTIVE,2026-10-08T10:05:00.000Z,2026-11-14T05:28:45.277Z,null,true
C014,Sneha P.,Mysore,sneha.p@example.com,9876543214,2026-03-20,ACTIVE,2026-10-09T08:00:00.000Z,2026-11-14T05:28:45.277Z,null,true
C015,Arjun,Chennai,arjun@example.com,9876543215,2026-03-25,ACTIVE,2026-10-08T10:15:00.000Z,2026-11-14T05:28:45.277Z,null,true


Pending events: 0
Pending events by operation:


operation,count


Current history records for affected customers:


customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current


History versions for affected customers:


customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current


In [0]:
from pyspark.sql import functions as F

ledger = spark.table("workspace.default.processed_cdc_events")
history = spark.table("workspace.default.dim_customers_history")

pending = ledger.filter(
    F.col("processing_status") == "PENDING"
)

display(pending)

# Get the actual CDC payload for each pending event
cdc = spark.table("workspace.default.silver_customer_cdc")

events = cdc.withColumn(
    "event_key",
    F.sha2(
        F.to_json(
            F.struct(
                "customer_id",
                "operation",
                "source_updated_at",
                "name",
                "city",
                "email",
                "phone",
                "signup_date",
                "status"
            )
        ),
        256
    )
)

# To get events payload.
pending_events = events.join(
    pending.select("event_key"),
    on="event_key",
    how="inner"
)
display(pending_events)

# Compare each event with the customer's history
comparison = (
    pending_events.alias("e")
    .join(
        history.alias("h"),
        F.col("e.customer_id") == F.col("h.customer_id"),
        "left"
    )
    .select(
        F.col("e.event_key"),
        F.col("e.customer_id"),
        F.col("e.operation"),
        F.col("e.source_updated_at"),
        F.col("e.city").alias("event_city"),
        F.col("e.status").alias("event_status"),
        F.col("h.city").alias("history_city"),
        F.col("h.status").alias("history_status"),
        F.col("h.is_current"),
        F.col("h.effective_start_date"),
        F.col("h.effective_end_date")
    )
)

display(comparison.orderBy("customer_id", "effective_start_date"))

event_key,customer_id,operation,source_updated_at,processing_status,processed_at


event_key,city,customer_id,email,name,operation,phone,signup_date,source_updated_at,status,_rescued_data,_ingested_at,_ingestion_date,_source_file


event_key,customer_id,operation,source_updated_at,event_city,event_status,history_city,history_status,is_current,effective_start_date,effective_end_date


In [0]:
# checking matching version, expired version, has current version
from pyspark.sql import functions as F

history = spark.table(
    "workspace.default.dim_customers_history"
)

eventss = pending_events.alias("e")
h = history.alias("h")

comparison = (
    eventss.join(
        h,
        F.col("e.customer_id") == F.col("h.customer_id"),
        "left"
    )
    .groupBy(
        F.col("e.event_key"),
        F.col("e.customer_id"),
        F.col("e.operation"),
        F.col("e.source_updated_at"),
        F.col("e.name").alias("event_name"),
        F.col("e.city").alias("event_city"),
        F.col("e.email").alias("event_email"),
        F.col("e.phone").alias("event_phone"),
        F.col("e.signup_date").alias("event_signup_date"),
        F.col("e.status").alias("event_status")
    )
    .agg(
        F.max(
            F.when(
                (F.col("h.is_current") == True) &
                (F.col("h.name") == F.col("e.name")) &
                (F.col("h.city") == F.col("e.city")) &
                (F.col("h.email") == F.col("e.email")) &
                (F.col("h.phone") == F.col("e.phone")) &
                (F.col("h.status") == F.col("e.status")) &
                (F.col("h.effective_start_date") == F.col("e.source_updated_at")),
                1
            ).otherwise(0)
        ).alias("matching_current_version"),

        F.max(
            F.when(
                (F.col("h.is_current") == False) &
                (F.col("h.effective_end_date") == F.col("e.source_updated_at")),
                1
            ).otherwise(0)
        ).alias("matching_expired_version"),

        F.max(
            F.when(F.col("h.is_current") == True, 1).otherwise(0)
        ).alias("has_current_version")
    )
)

display(comparison)

event_key,customer_id,operation,source_updated_at,event_name,event_city,event_email,event_phone,event_signup_date,event_status,matching_current_version,matching_expired_version,has_current_version


In [0]:
display(
    comparison.select(
        "customer_id",
        "operation",
        "matching_current_version",
        "matching_expired_version",
        "has_current_version"
    )
)

customer_id,operation,matching_current_version,matching_expired_version,has_current_version


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

LEDGER = "workspace.default.processed_cdc_events"

# Only events that passed our reconciliation checks
verified_events = comparison.filter(
    (
        (F.col("operation").isin("INSERT", "UPDATE")) &
        (F.col("matching_current_version") == 1)
    ) |
    (
        (F.col("operation") == "DELETE") &
        (F.col("matching_expired_version") == 1) &
        (F.col("has_current_version") == 0)
    )
)

verified_keys = verified_events.select("event_key").distinct()
display(verified_events)
ledger = DeltaTable.forName(spark, LEDGER)

(
    ledger.alias("t")
    .merge(
        verified_keys.alias("s"),
        "t.event_key = s.event_key"
    )
    .whenMatchedUpdate(
        condition="t.processing_status = 'PENDING'",
        set={
            "processing_status": "'PROCESSED'",
            "processed_at": "current_timestamp()"
        }
    )
    .execute()
)

print("Ledger reconciliation completed.")
display(spark.table(LEDGER))

event_key,customer_id,operation,source_updated_at,event_name,event_city,event_email,event_phone,event_signup_date,event_status,matching_current_version,matching_expired_version,has_current_version


Ledger reconciliation completed.


event_key,customer_id,operation,source_updated_at,processing_status,processed_at
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z


In [0]:
ledger_df = spark.table(
    "workspace.default.processed_cdc_events"
)

display(
    ledger_df.groupBy("processing_status").count()
)

processing_status,count
PROCESSED,4


In [0]:
display(
    history
    .filter(F.col("customer_id") == "C999")
    .select(
        "customer_id",
        "city",
        "status",
        "is_current",
        "effective_start_date",
        "effective_end_date"
    )
)

customer_id,city,status,is_current,effective_start_date,effective_end_date
C999,Chennai,ACTIVE,true,2026-10-25T10:02:00.000Z,null


In [0]:
before_count = (
    history
    .filter(F.col("customer_id") == "C999")
    .count()
)

print("History records before replay:", before_count)

display(
    spark.table("workspace.default.processed_cdc_events")
    .filter(F.col("customer_id") == "C999")
)



History records before replay: 1


event_key,customer_id,operation,source_updated_at,processing_status,processed_at
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z


In [0]:
replayed_event = events
display(replayed_event)
replay_check = (
    replayed_event.alias("e")
    .join(
        spark.table("workspace.default.processed_cdc_events")
        .alias("l"),
        on="event_key",
        how="left"
    )
    .select(
        "e.customer_id",
        "e.event_key",
        "l.processing_status"
    )
)
display(spark.table("workspace.default.processed_cdc_events"))

display(replay_check)

city,customer_id,email,name,operation,phone,signup_date,source_updated_at,status,_rescued_data,_ingested_at,_ingestion_date,_source_file,event_key
Coimbatore,C001,arun.kumar@example.com,ARUN,UPDATE,919876543210,2026-01-10,2026-10-25T10:00:00.000Z,ACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98
Bengaluru,C002,priya.sharma@example.com,Priya,UPDATE,919876543210,2026-01-12,2026-10-25T10:01:00.000Z,INACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3
Hyderabad,C003,karthik@example.com,Karthik,DELETE,919988776655,2026/01/15,2026-10-25T10:03:00.000Z,ACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece
Chennai,C999,test.customer999@example.com,Test Customer,INSERT,9000000099,2026-10-25,2026-10-25T10:02:00.000Z,ACTIVE,null,2026-10-24T15:53:38.472Z,2026-10-24,/Volumes/workspace/default/ecommerce_raw/cdc_incoming/customer_cdc_batch_001.json,6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42


event_key,customer_id,operation,source_updated_at,processing_status,processed_at
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z,PROCESSED,2026-11-09T07:14:46.917Z


customer_id,event_key,processing_status
C001,3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,PROCESSED
C002,b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,PROCESSED
C003,30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,PROCESSED
C999,6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,PROCESSED


In [0]:
# Already processed events
replayed_events = replay_check.filter(
    F.col("processing_status") == "PROCESSED"
)

# Events eligible for processing
eligible_events = replay_check.filter(
    (F.col("processing_status") == "PENDING") |
    F.col("processing_status").isNull()
)

print("Replayed events:", replayed_events.count())
print("Eligible events:", eligible_events.count())

Replayed events: 4
Eligible events: 0


In [0]:
display(
    spark.table("workspace.default.dim_customers_history")
    .filter(F.col("customer_id") == "C999")
    .select(
        "customer_id",
        "city",
        "status",
        "is_current",
        "effective_start_date",
        "effective_end_date"
    )
)

customer_id,city,status,is_current,effective_start_date,effective_end_date
C999,Chennai,ACTIVE,true,2026-10-25T10:02:00.000Z,null


In [0]:
#experiment
# recovery check for insert event.

TEST_HISTORY = "workspace.default.dim_customers_history_test"

(
    spark.table("workspace.default.dim_customers_history")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEST_HISTORY)
)

print("Test history table created successfully.")



Test history table created successfully.


In [0]:
test_history = spark.table(TEST_HISTORY)

display(
    test_history
    .filter(F.col("customer_id") == "C999")
    .select(
        "customer_id",
        "city",
        "status",
        "is_current",
        "effective_start_date",
        "effective_end_date"
    )
)

customer_id,city,status,is_current,effective_start_date,effective_end_date
C999,Chennai,ACTIVE,true,2026-10-25T10:02:00.000Z,null


In [0]:
from pyspark.sql import functions as F

# Get the original C999 event
retry_event = events.filter(
    F.col("customer_id") == "C999"
)

# Simulate a failed ledger update
simulated_ledger = (
    retry_event
    .select("event_key", "customer_id")
    .withColumn("processing_status", F.lit("PENDING"))
)

display(simulated_ledger)

event_key,customer_id,processing_status
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,PENDING


In [0]:
test_history = spark.table(TEST_HISTORY)

retry_event = events.filter(
    F.col("customer_id") == "C999"
).alias("e")

current_history = test_history.filter(
    (F.col("customer_id") == "C999") &
    (F.col("is_current") == True)
).alias("h")

recovery_check = retry_event.join(
    current_history,
    (F.col("e.customer_id") == F.col("h.customer_id")) &
    (F.col("e.city").eqNullSafe(F.col("h.city"))) &
    (F.col("e.status").eqNullSafe(F.col("h.status"))) &
    (F.col("e.source_updated_at") == F.col("h.effective_start_date")),
    "left"
).select(
    F.col("e.customer_id"),
    F.col("e.event_key"),
    F.col("e.operation"),
    F.col("h.is_current").alias("already_applied")
)

display(recovery_check)

customer_id,event_key,operation,already_applied
C999,6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,INSERT,true


In [0]:
reconciled_ledger = (
    simulated_ledger.alias("l")
    .join(
        recovery_check.select(
            "event_key",
            "already_applied"
        ).alias("r"),
        on="event_key",
        how="left"
    )
    .withColumn(
        "processing_status",
        F.when(
            F.col("already_applied") == True,
            F.lit("PROCESSED")
        ).otherwise(F.col("processing_status"))
    )
    .drop("already_applied")
)

display(reconciled_ledger)

event_key,customer_id,processing_status
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,PROCESSED


In [0]:
# Recovery for UPDATE and DELETE events, followed by out-of-order event handling.

test_history = spark.table(TEST_HISTORY)

display(
    test_history
    .filter(
        F.col("customer_id").isin(
            "C001", "C002", "C003", "C999"
        )
    )
    .select(
        "customer_id",
        "city",
        "status",
        "is_current",
        "effective_start_date",
        "effective_end_date"
    )
    .orderBy(
        "customer_id",
        "effective_start_date"
    )
)

customer_id,city,status,is_current,effective_start_date,effective_end_date
C001,Coimbatore,ACTIVE,true,2026-10-25T10:00:00.000Z,null
C001,Chennai,ACTIVE,false,2026-11-14T05:28:45.277Z,2026-10-25T10:00:00.000Z
C002,Bengaluru,INACTIVE,true,2026-10-25T10:01:00.000Z,null
C002,Bengaluru,ACTIVE,false,2026-11-14T05:28:45.277Z,2026-10-25T10:01:00.000Z
C003,Hyderabad,ACTIVE,false,2026-11-14T05:28:45.277Z,2026-10-25T10:03:00.000Z
C999,Chennai,ACTIVE,true,2026-10-25T10:02:00.000Z,null


In [0]:
from pyspark.sql import functions as F

# Event application = Was this particular change correctly reflected in history

e = events.alias("e")
h = test_history.alias("h")

recovery_status = (
    e.join(
        h,
        F.col("e.customer_id") == F.col("h.customer_id"),
        "left"
    )
    .groupBy(
        F.col("e.event_key"),
        F.col("e.customer_id"),
        F.col("e.operation"),
        F.col("e.source_updated_at")
    )
    .agg(
        F.max(
            F.when(
                (F.col("h.is_current") == True) &
                (F.col("h.effective_start_date") == F.col("e.source_updated_at")) &
                F.col("h.name").eqNullSafe(F.col("e.name")) &
                F.col("h.city").eqNullSafe(F.col("e.city")) &
                F.col("h.email").eqNullSafe(F.col("e.email")) &
                F.col("h.phone").eqNullSafe(F.col("e.phone")) &
                F.col("h.status").eqNullSafe(F.col("e.status")),
                1
            ).otherwise(0)
        ).alias("new_version_exists"),

        F.max(
            F.when(
                (F.col("h.is_current") == False) &
                (F.col("h.effective_end_date") == F.col("e.source_updated_at")),
                1
            ).otherwise(0)
        ).alias("old_version_expired"),

        F.max(
            F.when(
                F.col("h.is_current") == True, 1
            ).otherwise(0)
        ).alias("has_current_version")
    )
    .withColumn(
        "recovery_result",
        F.when(
            (F.col("operation").isin("INSERT", "UPDATE")) &
            (F.col("new_version_exists") == 1),
            "APPLIED"
        )
        .when(
            (F.col("operation") == "DELETE") &
            (F.col("old_version_expired") == 1) &
            (F.col("has_current_version") == 0),
            "APPLIED"
        )
        .otherwise("NOT_APPLIED_OR_REQUIRES_REVIEW")
    )
)

display(recovery_status)

event_key,customer_id,operation,source_updated_at,new_version_exists,old_version_expired,has_current_version,recovery_result
3f31c8643f307ee0c5bbbfbd6c52ecfbdeb890f5621206bb35507bfd71fc6b98,C001,UPDATE,2026-10-25T10:00:00.000Z,1,1,1,APPLIED
6e6a8fdc9a6e8ea19510609ae7162519525f0c3d62b2794bada2825876b64c42,C999,INSERT,2026-10-25T10:02:00.000Z,1,0,1,APPLIED
b3d924d2e30a69e6aaaf6f165cc036399f4aa2328f864069e0e14015124a82d3,C002,UPDATE,2026-10-25T10:01:00.000Z,1,1,1,APPLIED
30d82b1591e63e4da295a01f50349361992411da530a0565a465ec3f5cdd9ece,C003,DELETE,2026-10-25T10:03:00.000Z,0,1,0,APPLIED


In [0]:
display(
    recovery_status.select(
        "customer_id",
        "operation",
        "new_version_exists",
        "old_version_expired",
        "has_current_version",
        "recovery_result"
    ).orderBy("customer_id")
)

customer_id,operation,new_version_exists,old_version_expired,has_current_version,recovery_result
C001,UPDATE,1,1,1,APPLIED
C002,UPDATE,1,1,1,APPLIED
C003,DELETE,0,1,0,APPLIED
C999,INSERT,1,0,1,APPLIED


In [0]:
# out of order handling

test_history = spark.table(TEST_HISTORY)

display(
    test_history
    .filter(
        (F.col("customer_id") == "C001") &
        (F.col("is_current") == True)
    )
    .select(
        "customer_id",
        "city",
        "status",
        "effective_start_date",
        "is_current"
    )
)

test_history.printSchema()

customer_id,city,status,effective_start_date,is_current
C001,Coimbatore,ACTIVE,2026-10-25T10:00:00.000Z,true


root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- effective_start_date: timestamp (nullable = true)
 |-- effective_end_date: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)



In [0]:
from pyspark.sql import functions as F

# Get current C001 record
base = (
    spark.table(TEST_HISTORY)
    .filter(
        (F.col("customer_id") == "C001") &
        (F.col("is_current") == True)
    )
)

display(base)

# Newer event: Bengaluru
newer_event = (
    base.select(
        "customer_id",
        "name",
        "email",
        "phone",
        "signup_date",
        "status"
    )
    .withColumn("city", F.lit("Bengaluru"))
    .withColumn(
        "operation", F.lit("UPDATE")
    )
    .withColumn(
        "source_updated_at",
        F.to_timestamp(F.lit("2026-10-25 10:05:00"))
    )
)

# Older event: Chennai
older_event = (
    base.select(
        "customer_id",
        "name",
        "email",
        "phone",
        "signup_date",
        "status"
    )
    .withColumn("city", F.lit("Chennai"))
    .withColumn(
        "operation", F.lit("UPDATE")
    )
    .withColumn(
        "source_updated_at",
        F.to_timestamp(F.lit("2026-10-25 10:02:00"))
    )
)

# Simulate arrival order: newer first, older second
out_of_order_events = newer_event.unionByName(older_event)

display(
    out_of_order_events.select(
        "customer_id",
        "operation",
        "city",
        "source_updated_at"
    )
)

customer_id,name,city,email,phone,signup_date,status,source_updated_at,effective_start_date,effective_end_date,is_current
C001,ARUN,Coimbatore,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,null,2026-10-25T10:00:00.000Z,null,true


customer_id,operation,city,source_updated_at
C001,UPDATE,Bengaluru,2026-10-25T10:05:00.000Z
C001,UPDATE,Chennai,2026-10-25T10:02:00.000Z


In [0]:
current_c001 = (
    spark.table(TEST_HISTORY)
    .filter(
        (F.col("customer_id") == "C001") &
        (F.col("is_current") == True)
    )
    .select(
        "customer_id",
        F.col("effective_start_date").alias("current_timestamp")
    )
)

newer_event_to_process = (
    newer_event.alias("e")
    .join(
        current_c001.alias("h"),
        on="customer_id",
        how="inner"
    )
    .filter(
        F.col("e.source_updated_at") >
        F.col("h.current_timestamp")
    )
    .select("e.*")
)

display(
    newer_event_to_process.select(
        "customer_id",
        "city",
        "source_updated_at"
    )
)

customer_id,city,source_updated_at
C001,Bengaluru,2026-10-25T10:05:00.000Z


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Prepare the expiration record
expire_rows = (
    newer_event_to_process
    .withColumn("_merge_key", F.col("customer_id"))
    .withColumn("action", F.lit("EXPIRE"))
)

display(expire_rows)

# Prepare the new version
insert_rows = (
    newer_event_to_process
    .withColumn("_merge_key", F.lit(None).cast("string"))
    .withColumn("action", F.lit("INSERT"))
)

display(expire_rows)


# Combine both actions
staged_events = expire_rows.unionByName(insert_rows)

target = DeltaTable.forName(spark, TEST_HISTORY)

(
    target.alias("t")
    .merge(
        staged_events.alias("s"),
        """
        t.customer_id = s._merge_key
        AND t.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="s.action = 'EXPIRE'",
        set={
            "is_current": "false",
            "effective_end_date": "s.source_updated_at"
        }
    )
    .whenNotMatchedInsert(
        condition="s.action = 'INSERT'",
        values={
            "customer_id": "s.customer_id",
            "name": "s.name",
            "city": "s.city",
            "email": "s.email",
            "phone": "s.phone",
            "signup_date": "s.signup_date",
            "status": "s.status",
            "source_updated_at": "s.source_updated_at",
            "effective_start_date": "s.source_updated_at",
            "effective_end_date": "CAST(NULL AS TIMESTAMP)",
            "is_current": "true"
        }
    )
    .execute()
)

print("Newer CDC event applied successfully.")

customer_id,name,email,phone,signup_date,status,city,operation,source_updated_at,_merge_key,action
C001,ARUN,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,Bengaluru,UPDATE,2026-10-25T10:05:00.000Z,C001,EXPIRE


customer_id,name,email,phone,signup_date,status,city,operation,source_updated_at,_merge_key,action
C001,ARUN,arun.kumar@example.com,919876543210,2026-01-10,ACTIVE,Bengaluru,UPDATE,2026-10-25T10:05:00.000Z,C001,EXPIRE


Newer CDC event applied successfully.


In [0]:
display(
    spark.table(TEST_HISTORY)
    .filter(F.col("customer_id") == "C001")
    .select(
        "customer_id",
        "city",
        "source_updated_at",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .orderBy("effective_start_date")
)

customer_id,city,source_updated_at,effective_start_date,effective_end_date,is_current
C001,Coimbatore,null,2026-10-25T10:00:00.000Z,2026-10-25T10:05:00.000Z,false
C001,Bengaluru,2026-10-25T10:05:00.000Z,2026-10-25T10:05:00.000Z,null,true
C001,Chennai,2026-10-08T09:00:00.000Z,2026-11-14T05:28:45.277Z,2026-10-25T10:00:00.000Z,false


In [0]:
display(
    spark.table(TEST_HISTORY)
    .filter(F.col("customer_id") == "C001")
    .select(
        "customer_id",
        "city",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .orderBy("effective_start_date")
)

customer_id,city,effective_start_date,effective_end_date,is_current
C001,Coimbatore,2026-10-25T10:00:00.000Z,2026-10-25T10:05:00.000Z,false
C001,Bengaluru,2026-10-25T10:05:00.000Z,null,true
C001,Chennai,2026-11-14T05:28:45.277Z,2026-10-25T10:00:00.000Z,false
